# Congress stock trading API: House members' trades as JSON

Every stock trade House members report in their periodic transaction reports, searchable by member, ticker, party, amount and date. Python and JavaScript.

Read it with charts at https://www.quantisnow.com/sec-api/guides/congress-stock-trading-api. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

The STOCK Act gives members of Congress 45 days to report a trade over $1,000. The House clerk publishes the reports as PDFs, some typed and some scanned. The [Congress trades endpoint](https://www.quantisnow.com/sec-api/congress-trades) reads each one into records with the member, the asset, the ticker, the trade type, the amount range and both dates.

## Who traded a stock

### Which House members traded Nvidia this year?

Everyone who bought or sold one stock, with the dollar range each reported.

```shell
curl -X POST https://www.quantisnow.com/api/v1/congress/trades \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "ticker:NVDA AND tradedOn:[2026-01-01 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"tradedOn": {"order": "desc"}}]
}'
```

Response (real, first 2 of 45):

```json
{
  "total": {
    "value": 45,
    "relation": "eq"
  },
  "data": [
    {
      "id": "20036855-6",
      "reportId": "20036855",
      "chamber": "house",
      "member": {
        "name": "Michael A. Rulli",
        "bioguideId": "R000619",
        "state": "OH",
        "district": "OH06",
        "party": "R"
      },
      "owner": "self",
      "ticker": "NVDA",
      "asset": "NVIDIA Corporation - Common Stock (NVDA)",
      "assetType": "ST",
      "type": "partial_sale",
      "tradedOn": "2026-10-28",
      "notifiedOn": "2026-10-28",
      "filedOn": "2026-11-18",
      "amount": {
        "min": 1001,
        "max": 15000
      },
      "description": "Subholding Of: Merrill Lynch SIMPLE Managed Account",
      "reportUrl": "https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2026/20036855.pdf"
    },
    {
      "id": "20035778-3",
      "reportId": "20035778",
      "chamber": "house",
      "member": {
        "name": "Maria Elvira Salazar",
        "bioguideId": "S000168",
        "state": "FL",
        "district": "FL27",
        "party": "R"
      },
      "owner": "self",
      "ticker": "NVDA",
      "asset": "NVIDIA Corporation - Common Stock (NVDA)",
      "assetType": "ST",
      "type": "purchase",
      "tradedOn": "2026-10-23",
      "notifiedOn": "2026-11-01",
      "filedOn": "2026-11-05",
      "amount": {
        "min": 1001,
        "max": 15000
      },
      "description": "Filing Status: New; Subholding Of: UBS Brokerage Account",
      "reportUrl": "https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2026/20035778.pdf"
    }
  ]
}
```

## The largest purchases

### Which purchases over $1 million did members report in the last 90 days?

The largest disclosed buys, where the low end of the range alone is seven figures.

```shell
curl -X POST https://www.quantisnow.com/api/v1/congress/trades \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "type:purchase AND amount.min:>=1000000 AND filedOn:[2026-08-26 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"amount.min": {"order": "desc"}}]
}'
```

Response (real, first 2 of 1):

```json
{
  "total": {
    "value": 1,
    "relation": "eq"
  },
  "data": [
    {
      "id": "20035700-1",
      "reportId": "20035700",
      "chamber": "house",
      "member": {
        "name": "Darrell Issa",
        "bioguideId": "I000056",
        "state": "CA",
        "district": "CA48",
        "party": "R"
      },
      "owner": "self",
      "ticker": null,
      "asset": "US Treasury Note 3.375% due 09/15/28 CUSIP 91282CNY3",
      "assetType": "GS",
      "type": "purchase",
      "tradedOn": "2026-10-08",
      "notifiedOn": "2026-10-09",
      "filedOn": "2026-10-22",
      "amount": {
        "min": 5000001,
        "max": 25000000
      },
      "description": "Purchase of US Treasury Bond with $10 million face value",
      "reportUrl": "https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2026/20035700.pdf"
    }
  ]
}
```

Amounts are ranges because that's what the form asks for: $1,001 to $15,000, $15,001 to $50,000, up to over $50 million. `amount.min` and `amount.max` hold the ends, so `amount.min:>=1000000` means at least a million.

## In Python

In [ ]:
import os, requests

BASE = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]

def search(endpoint, query, size=50, sort=None, start=0):
    body = {"query": query, "size": size, "from": start}
    if sort:
        body["sort"] = sort
    res = requests.post(f"{BASE}/{endpoint}", json=body,
                        headers={"Authorization": f"Bearer {KEY}"}, timeout=30)
    res.raise_for_status()
    return res.json()
from collections import Counter

page = search("congress/trades", "type:purchase AND tradedOn:[2026-08-01 TO 2026-11-23]", size=50)
by_ticker = Counter(t["ticker"] for t in page["data"] if t["ticker"])
print(by_ticker.most_common(10))

## In JavaScript

```javascript
const BASE = "https://www.quantisnow.com/api/v1";

async function search(endpoint, query, { size = 50, sort, from = 0 } = {}) {
  const res = await fetch(BASE + "/" + endpoint, {
    method: "POST",
    headers: {
      Authorization: "Bearer " + process.env.QUANTISNOW_API_KEY,
      "Content-Type": "application/json",
    },
    body: JSON.stringify({ query, size, from, sort }),
  });
  if (!res.ok) throw new Error((await res.json()).error.message);
  return res.json();
}

// One member's trades, newest first
const page = await search("congress/trades", 'member.name:"Michael A. Rulli"', { sort: "tradedOn:desc" });
for (const t of page.data) console.log(t.tradedOn, t.type, t.ticker, t.amount.min + "-" + t.amount.max);
```

The gap between `tradedOn` and `filedOn` is often weeks. Sort by `filedOn` to see what became public most recently, and by `tradedOn` to see when the trade happened.